<a href="https://colab.research.google.com/github/gabrielebirri/CLAMA_Project/blob/main/Pipeline_Driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Melanoma identification through deep transfer learning**  

Author: Gabriele Birri  
Date: March 2026


## Introduction  
### Abstract
The objective of this experimental research is to conduct a comparative analysis of diverse Convolutional Neural Network (CNN) architectures to assess their performance in the automated classification of cutaneous lesions as benign or malignant (melanoma). Given the inherent constraints of medical imaging datasets, a transfer learning methodology is adopted. Specifically, three prominent architectures, pre-trained on the ImageNet database, are employed as feature extractors to evaluate their diagnostic efficacy. To ensure experimental integrity and mitigate methodological biases, a standardized training and evaluation pipeline is maintained across all models.

The architectures are adapted for binary classification by substituting the original global classifier or fully connected layer with a task-specific head. The optimization follows a two-phase fine-tuning strategy: an initial stage where the convolutional backbone remains frozen to stabilize the new classifier's weights, followed by a selective unfreezing of deeper layers to facilitate domain-specific feature adaptation. This dual-stage approach is designed to enhance task specialization while preserving the generalization capabilities of the pre-trained weights. Furthermore, a grad-CAM technique is implemented to gain a sensible evidence of the spatial attention of the model.

The study utilizes a dataset of approximately 10,000 dermoscopic images, sourced from a publicly available repository and balanced between benign and malignant classes. The performance of the respective architectures is rigorously quantified using standard clinical evaluation metrics, providing a comprehensive perspective on the suitability of different CNN backbones for computer-aided diagnosis in dermatology.  

**Dataset used:** https://www.kaggle.com/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images

### The Significance of early Melanoma Detection
Melanoma is a malignant neoplasm arising from the oncogenic transformation of melanocytes, the pigment-producing cells located in the basal layer of the epidermis. While it accounts for a minority of skin cancer cases, it is responsible for the vast majority of skin cancer-related fatalities due to its aggressive nature and high metastatic potential. Precision and speed in diagnosis are paramount; according to the American Cancer Society (2025), the 5-year survival rate exceeds 99% when detected at a localized stage, but plummets to approximately 35% once distant metastasis occurs. Consequently, developing automated, highly accurate diagnostic tools is a critical priority in reducing clinical variability and improving patient outcomes through early intervention.

**Key References**
* American Cancer Society. (2025). Cancer Facts & Figures 2025. Atlanta: American Cancer Society.  

* Haenssle, M. A., et al. (2018). Man against machine: diagnostic performance of a deep learning convolutional neural network for dermoscopic melanoma recognition in comparison to 58 dermatologists. Annals of Oncology, 29(8), 1836-1842.  

* World Health Organization. (2024). Global Cancer Statistics: Skin Melanoma.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Detected environment: Colab")
    print("Proceeding to download grad-cam library")
    !pip install grad-cam -q
    !git clone https://github.com/gabrielebirri/CLAMA_Project

else:
    print("Detected environment: Local")

In [ ]:
## IMPORT PACKAGES ##

# Utility
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from tqdm import tqdm
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
import copy

# PyTorch
import torch
from torch import nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


print("all clear")

In [ ]:
# Setting up the agnostic device code
if torch.backends.mps.is_available():
    device = "mps"  # For MPS acceleration
elif torch.cuda.is_available():
    device = "cuda" # For CUDA acceleration
else:
    device = "cpu"

print(f"Device: {device}")

# Pre-processing of the image data

**Data augmentation**

To improve the generalization capability of the model and mitigate overfitting, a data augmentation strategy is applied to the training dataset. The selected transformations are carefully chosen to preserve the clinically relevant diagnostic features of melanoma while introducing controlled variability in the input images.

The transformations employed include resizing, horizontal flipping, vertical flipping and rotation. Image resizing standardizes the input dimensions to match the architectural requirements of the neural network, ensuring consistent tensor shapes across the dataset. Horizontal and vertical flips are applied to simulate variations in lesion orientation. Since melanoma diagnosis is independent of the spatial orientation of the lesion, these transformations do not alter essential diagnostic criteria such as asymmetry, border irregularity, color heterogeneity, or structural patterns. Similarly, controlled rotations are introduced to further increase rotational invariance, reflecting the fact that skin lesions may be photographed from different angles without affecting their pathological characteristics.

It's important to note that no transformations that could distort morphological or chromatic properties are applied. The selected augmentations preserve the intrinsic geometrical and textural features that are critical for melanoma identification, thereby maintaining clinical validity while enhancing model robustness.

**Image normalization**

To minimize the objective function more efficiently and maintain architectural consistency with the pre-trained EfficientNetV2-S backbone, a Z-score normalization was applied. This transformation centers the pixel intensity distribution around zero and scales it to unit variance, preventing gradient instability and ensuring that the visual features extracted are robust to varying lighting conditions of the source mobile devices.

**Dataset splitting**

The dataset is splitted into three subsets (of approximately the sizes indicated):

* Training (80%)  

* Validation (10%)  

* Test (10%)  

The training set is used to optimize the model parameters through backpropagation. The validation set is employed during development to tune hyperparameters, monitor generalization performance, and implement strategies such as early stopping. The test set, which remains completely unseen during training and validation, is used exclusively for the final performance evaluation to provide an unbiased estimate of the model’s predictive capability on new data.
This structured splitting ensures methodological rigor and prevents information leakage, thereby supporting reliable assessment of the model’s real-world applicability.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Detected environment: Colab")
    # Downloading the dataset from Kaggle
    root = kagglehub.dataset_download("hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images")
    train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
    test_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/test'

else:
    print("Detected environment: Local")
    root = './melanoma_cancer_dataset'
    train_path = os.path.join(root, "train")
    test_path = os.path.join(root, "test")